In [1]:
import numpy as np
from py_group_sequential_designs import generate_boundaries as bd
from py_group_sequential_designs import simulate as sim
import timeit

In [2]:
def reverse_to_boundaries(params, K):
    params = np.asarray(params).flatten()
    c = params[0]

    delta_u = params[1::2][::-1]
    delta_l = params[2::2][::-1]

    upper_bounds = np.array([c + np.sum(delta_u[k:]) for k in range(K)])
    lower_bounds = np.array([c - np.sum(delta_l[k:]) for k in range(K)])

    return upper_bounds, lower_bounds

def boundaries_to_reverse(upper_bounds, lower_bounds):
    upper_bounds = np.asarray(upper_bounds)
    lower_bounds = np.asarray(lower_bounds)

    K = len(upper_bounds)
    c = upper_bounds[-1]

    delta_u = np.diff(upper_bounds[::-1])
    delta_l = np.diff(lower_bounds)[::-1]

    increments = np.empty(2 * (K - 1))
    increments[0::2] = delta_u
    increments[1::2] = delta_l

    return np.concatenate([[c], increments])

In [3]:
def reverse_to_boundaries_new(params, K):
    params = np.asarray(params).flatten()
    c = params[0]

    # delta_u and delta_l are already sliced and reversed
    delta_u = params[1::2][::-1]
    delta_l = params[2::2][::-1]

    u_pads = np.concatenate([[0], delta_u[::-1]])
    l_pads = np.concatenate([[0], delta_l[::-1]])

    # np.cumsum(x[::-1])[::-1] efficiently computes suffix sums from left-to-right
    upper_bounds = c + np.cumsum(u_pads)[::-1]
    lower_bounds = c - np.cumsum(l_pads)[::-1]

    return upper_bounds, lower_bounds

def boundaries_to_reverse_new(upper_bounds, lower_bounds):
    upper_bounds = np.asarray(upper_bounds)
    lower_bounds = np.asarray(lower_bounds)

    c = upper_bounds[-1]

    delta_u = np.diff(upper_bounds[::-1])
    delta_l = np.diff(lower_bounds)[::-1]

    increments = np.empty(2 * len(delta_u), dtype=upper_bounds.dtype)
    increments[0::2] = delta_u
    increments[1::2] = delta_l

    return np.concatenate([[c], increments])

In [4]:
orig_time = timeit.timeit(
    stmt="reverse_to_boundaries([1.83560794, 0.03785157, 0.71153224, 0.24611797, 1.12407571], 3)",
    globals=globals(),
    number=10000
)

# Time the optimized function
opt_time = timeit.timeit(
    stmt="reverse_to_boundaries_new([1.83560794, 0.03785157, 0.71153224, 0.24611797, 1.12407571], 3)",
    globals=globals(),
    number=10000
)

# --- 5. RESULTS DISPLAY ---
print(f"Original Code Total Time  : {orig_time:.4f} seconds")
print(f"Optimized Code Total Time : {opt_time:.4f} seconds")
print(f"Speedup Factor            : {orig_time / opt_time:.1f}x faster")

Original Code Total Time  : 0.0734 seconds
Optimized Code Total Time : 0.0455 seconds
Speedup Factor            : 1.6x faster


In [5]:
def max_ess_derivative(
        delta_start=0,
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        variance=1,
        epsilon=1e-2):

    delta_stop = variance * 5.
    
    # step size to calculate the derivative (slope)
    h = 1e-5

    def get_ess(delta):
        _, _, ess = sim.group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_patients,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = delta,
            variance = variance
        )
        return ess

    # bisection loop
    while (delta_stop - delta_start) > epsilon:
        midpoint = (delta_start + delta_stop) / 2.0
        
        # sample slightly to the left and right of the midpoint to get the slope
        ess_left = get_ess(midpoint - h)
        ess_right = get_ess(midpoint + h)
        
        slope = ess_right - ess_left

        if slope > 0:
            # slope is positive: we are climbing up the left side of the hill.
            # the peak must be to the right.
            delta_start = midpoint
        else:
            # slope is negative: we are sliding down the right side of the hill.
            # The peak must be to the left.
            delta_stop = midpoint

    # Return the final optimized ESS at the peak midpoint
    return get_ess((delta_start + delta_stop) / 2.0)

In [6]:
def max_ess(
        delta_start=0,
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        variance=1):

    # epsilon precision for max ESS calculated
    epsilon = 1e-4

    # delta_stop based on the variance (5x the variance)
    delta_stop = variance * 5
    
    # random starting values of ess
    ess_delta_start = 10
    ess_delta_stop = 0
    
    # while the error is greater than desired precision
    while abs(ess_delta_start - ess_delta_stop) > epsilon:

        # simulate the trial under delta_start
        _, _, ess_delta_start = sim.group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_patients,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = delta_start,
            variance = variance
        )
    
        # simulate trial under delta_stop
        _, _, ess_delta_stop = sim.group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_patients,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = delta_stop,
            variance = variance
        )
        
        if ess_delta_start >= ess_delta_stop:
            delta_stop = (delta_start + delta_stop)/2 
        else:
            delta_start = (delta_start + delta_stop)/2

    return ess_delta_start

In [7]:
max_ess_derivative()

43.83197551169668

In [8]:
max_ess()

43.832718378823984

In [9]:
orig_time = timeit.timeit(
    stmt="max_ess()",
    globals=globals(),
    number=1000
)

# Time the optimized function
opt_time = timeit.timeit(
    stmt="max_ess_derivative()",
    globals=globals(),
    number=1000
)

# --- 5. RESULTS DISPLAY ---
print(f"Original Code Total Time  : {orig_time:.4f} seconds")
print(f"Optimized Code Total Time : {opt_time:.4f} seconds")
print(f"Speedup Factor            : {orig_time / opt_time:.1f}x faster")

Original Code Total Time  : 1.8824 seconds
Optimized Code Total Time : 1.3574 seconds
Speedup Factor            : 1.4x faster


In [10]:
def find_sample_size_new(
        power_target=0.9,
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1,
        epsilon=1e-2): # Explicitly define the precision you want

    n_min = 2.0
    n_max = 1000.0

    while (n_max - n_min) > epsilon:
        n_mid = (n_min + n_max) / 2.0

        _, power_mid, _ = sim.group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_mid,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = alt_hypothesis,
            variance = variance
        )

        if power_mid >= power_target:
            n_max = n_mid
        else:
            n_min = n_mid

    return [n_max, power_mid]

In [11]:
def find_sample_size(
        power_target=0.9,
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1):

    # initial sample sizes
    n_patients_min = 2
    n_patients_max = 5000

    _, power_max, _ = sim.group_sequential_designs(
        n_analyses = n_analyses,
        upper_bounds = upper_bounds,
        lower_bounds = lower_bounds,
        n_patients = n_patients_max,
        null_hypothesis = null_hypothesis,
        alt_hypothesis = alt_hypothesis,
        variance = variance
    )
    
    if power_max < power_target:
        return None

    while n_patients_max - n_patients_min > 0.5:

        n_mid = (n_patients_min + n_patients_max) / 2

        _, power_mid, _ = sim.group_sequential_designs(
            n_analyses = n_analyses,
            upper_bounds = upper_bounds,
            lower_bounds = lower_bounds,
            n_patients = n_mid,
            null_hypothesis = null_hypothesis,
            alt_hypothesis = alt_hypothesis,
            variance = variance
        )

        if power_mid >= power_target:
            n_patients_max = n_mid
        else:
            n_patients_min = n_mid

    return [n_patients_min, power_mid]

In [12]:
find_sample_size()

[23.9638671875, 0.9014776486397186]

In [13]:
find_sample_size_new()

[24.119064331054688, 0.9000517845660573]

In [14]:
orig_time = timeit.timeit(
    stmt="find_sample_size()",
    globals=globals(),
    number=1000
)

# Time the optimized function
opt_time = timeit.timeit(
    stmt="find_sample_size_new()",
    globals=globals(),
    number=1000
)

# --- 5. RESULTS DISPLAY ---
print(f"Original Code Total Time  : {orig_time:.4f} seconds")
print(f"Optimized Code Total Time : {opt_time:.4f} seconds")
print(f"Speedup Factor            : {orig_time / opt_time:.1f}x faster")

Original Code Total Time  : 1.1860 seconds
Optimized Code Total Time : 1.2368 seconds
Speedup Factor            : 1.0x faster
